# AgTech: Automação de Precisão com Sistema Especialista e Gemini

**Tema:** Sistemas para drones ou robôs agrícolas que identificam pragas e gerenciam recursos hídricos.  
**Abordagem escolhida:** **Opção B — Sistema Especialista**.  
**Justificativa técnica:** o agente recebe saídas de uma Rede Neural de Visão Computacional, como praga detectada, confiança e percentual de dano foliar, além de dados ambientais/hídricos. A decisão técnica é feita por regras explícitas, enquanto o Gemini atua apenas como camada de **Análise Interpretativa**.

> Observação: este notebook usa dados simulados para representar a saída de uma Rede Neural. Em uma versão futura, essa entrada pode ser substituída por um modelo real treinado em imagens de lavoura.


## 1. Arquitetura lógica do agente

Fluxo principal:

```text
Imagem do drone/robô
        ↓
Rede Neural de Visão Computacional
        ↓
Saída simulada: praga_detectada, confiança, dano_foliar
        ↓
Sensores/contexto: umidade_solo, chuva_prevista, reservatório, temperatura
        ↓
Sistema Especialista
        ↓
Decisão técnica: risco de praga + ação hídrica + prioridade operacional
        ↓
Gemini API
        ↓
Explicação em linguagem natural + alertas estratégicos
```

O Gemini **não toma a decisão técnica**. Ele recebe o resultado do Sistema Especialista e transforma a saída em uma explicação clara para o operador.


In [ ]:
# Instalação das bibliotecas necessárias no Google Colab
!pip -q install google-genai pandas


In [ ]:
import os
import json
import pandas as pd
from dataclasses import dataclass, asdict
from getpass import getpass


## 2. Entradas do agente

As entradas abaixo simulam:

- **Saída da Rede Neural:** praga detectada, confiança da classificação e percentual estimado de dano foliar.
- **Dados de sensores e contexto:** umidade do solo, chuva prevista, nível do reservatório e temperatura.

Esses dados representam diferentes talhões de uma lavoura de soja.


In [ ]:
cenarios = [
    {
        "talhao": "T-01",
        "cultura": "Soja",
        "praga_detectada": "lagarta",
        "confianca_visao": 0.92,
        "dano_foliar_pct": 31,
        "umidade_solo_pct": 24,
        "chuva_prevista_mm": 1,
        "nivel_reservatorio_pct": 68,
        "temperatura_c": 33.5
    },
    {
        "talhao": "T-02",
        "cultura": "Soja",
        "praga_detectada": "vaquinha",
        "confianca_visao": 0.76,
        "dano_foliar_pct": 14,
        "umidade_solo_pct": 41,
        "chuva_prevista_mm": 0,
        "nivel_reservatorio_pct": 54,
        "temperatura_c": 31.0
    },
    {
        "talhao": "T-03",
        "cultura": "Soja",
        "praga_detectada": "sem_praga",
        "confianca_visao": 0.88,
        "dano_foliar_pct": 2,
        "umidade_solo_pct": 52,
        "chuva_prevista_mm": 12,
        "nivel_reservatorio_pct": 80,
        "temperatura_c": 27.0
    },
    {
        "talhao": "T-04",
        "cultura": "Soja",
        "praga_detectada": "lagarta",
        "confianca_visao": 0.61,
        "dano_foliar_pct": 8,
        "umidade_solo_pct": 22,
        "chuva_prevista_mm": 0,
        "nivel_reservatorio_pct": 21,
        "temperatura_c": 34.0
    }
]

pd.DataFrame(cenarios)


## 3. Base de conhecimento do Sistema Especialista

### Regras para pragas

| Condição | Risco | Ação |
|---|---|---|
| Praga detectada, confiança ≥ 0,80 e dano foliar ≥ 25% | Crítico | Intervenção localizada e alerta técnico |
| Praga detectada, confiança ≥ 0,70 e dano foliar entre 10% e 25% | Alto | Inspeção prioritária e manejo localizado |
| Praga detectada, confiança ≥ 0,50 ou dano foliar entre 5% e 10% | Médio | Monitoramento reforçado |
| Sem praga ou dano muito baixo | Baixo | Monitoramento normal |

### Regras para recursos hídricos

| Condição | Ação hídrica |
|---|---|
| Umidade < 30%, chuva < 5 mm e reservatório ≥ 30% | Irrigar |
| Umidade < 30%, chuva < 5 mm e reservatório < 30% | Irrigação restrita e alerta de recurso hídrico |
| Umidade entre 30% e 45%, chuva < 3 mm e temperatura > 32 ºC | Irrigação moderada |
| Chuva prevista ≥ 10 mm | Suspender irrigação |
| Caso contrário | Manter monitoramento |


In [ ]:
def classificar_praga(entrada):
    praga = entrada["praga_detectada"]
    confianca = entrada["confianca_visao"]
    dano = entrada["dano_foliar_pct"]

    if praga != "sem_praga" and confianca >= 0.80 and dano >= 25:
        return {
            "risco_praga": "crítico",
            "acao_praga": "intervenção localizada imediata e alerta ao responsável técnico",
            "regra_acionada": "R1: praga confirmada com alta confiança e dano foliar severo"
        }

    if praga != "sem_praga" and confianca >= 0.70 and 10 <= dano < 25:
        return {
            "risco_praga": "alto",
            "acao_praga": "inspeção prioritária e manejo localizado",
            "regra_acionada": "R2: praga confirmada com dano foliar moderado"
        }

    if praga != "sem_praga" and (confianca >= 0.50 or 5 <= dano < 10):
        return {
            "risco_praga": "médio",
            "acao_praga": "monitoramento reforçado e nova coleta de imagens",
            "regra_acionada": "R3: indício de praga ou dano inicial"
        }

    return {
        "risco_praga": "baixo",
        "acao_praga": "monitoramento normal",
        "regra_acionada": "R4: ausência de praga relevante ou dano foliar baixo"
    }


def classificar_hidrico(entrada):
    umidade = entrada["umidade_solo_pct"]
    chuva = entrada["chuva_prevista_mm"]
    reservatorio = entrada["nivel_reservatorio_pct"]
    temperatura = entrada["temperatura_c"]

    if chuva >= 10:
        return {
            "risco_hidrico": "baixo",
            "acao_hidrica": "suspender irrigação e reavaliar após chuva",
            "regra_acionada_hidrica": "H4: chuva prevista suficiente"
        }

    if umidade < 30 and chuva < 5 and reservatorio >= 30:
        return {
            "risco_hidrico": "alto",
            "acao_hidrica": "acionar irrigação controlada",
            "regra_acionada_hidrica": "H1: solo seco, baixa chuva e reservatório adequado"
        }

    if umidade < 30 and chuva < 5 and reservatorio < 30:
        return {
            "risco_hidrico": "crítico",
            "acao_hidrica": "irrigação restrita, priorização de áreas críticas e alerta de reservatório baixo",
            "regra_acionada_hidrica": "H2: solo seco e reservatório baixo"
        }

    if 30 <= umidade <= 45 and chuva < 3 and temperatura > 32:
        return {
            "risco_hidrico": "médio",
            "acao_hidrica": "irrigação moderada preventiva",
            "regra_acionada_hidrica": "H3: umidade intermediária com alta temperatura"
        }

    return {
        "risco_hidrico": "baixo",
        "acao_hidrica": "manter monitoramento hídrico",
        "regra_acionada_hidrica": "H5: condições hídricas dentro do aceitável"
    }


def definir_prioridade(decisao_praga, decisao_hidrica):
    risco_praga = decisao_praga["risco_praga"]
    risco_hidrico = decisao_hidrica["risco_hidrico"]

    if risco_praga == "crítico":
        return "prioridade máxima: tratar foco de praga antes de ampliar irrigação"
    if risco_hidrico == "crítico":
        return "prioridade alta: preservar recursos hídricos e priorizar áreas com maior estresse"
    if risco_praga == "alto" or risco_hidrico == "alto":
        return "prioridade alta: executar ação localizada no talhão"
    if risco_praga == "médio" or risco_hidrico == "médio":
        return "prioridade média: acompanhar evolução e coletar novos dados"
    return "prioridade baixa: rotina de monitoramento"


def sistema_especialista(entrada):
    decisao_praga = classificar_praga(entrada)
    decisao_hidrica = classificar_hidrico(entrada)
    prioridade = definir_prioridade(decisao_praga, decisao_hidrica)

    return {
        "entrada": entrada,
        "decisao_praga": decisao_praga,
        "decisao_hidrica": decisao_hidrica,
        "prioridade_operacional": prioridade
    }


In [ ]:
logs = [sistema_especialista(cenario) for cenario in cenarios]

resumo = []
for log in logs:
    entrada = log["entrada"]
    resumo.append({
        "talhao": entrada["talhao"],
        "praga": entrada["praga_detectada"],
        "confianca_visao": entrada["confianca_visao"],
        "dano_foliar_pct": entrada["dano_foliar_pct"],
        "risco_praga": log["decisao_praga"]["risco_praga"],
        "acao_praga": log["decisao_praga"]["acao_praga"],
        "risco_hidrico": log["decisao_hidrica"]["risco_hidrico"],
        "acao_hidrica": log["decisao_hidrica"]["acao_hidrica"],
        "prioridade": log["prioridade_operacional"]
    })

df_logs = pd.DataFrame(resumo)
df_logs


In [ ]:
# Exemplo de log detalhado para documentação no GitHub
print(json.dumps(logs[0], ensure_ascii=False, indent=2))


## 4. Integração com a API do Gemini

Nesta etapa, o Gemini recebe o log do Sistema Especialista e gera:

1. Explicação em linguagem natural.
2. Justificativa da decisão.
3. Alertas estratégicos.
4. Sugestões de próximos passos.

**Importante:** a decisão técnica já foi definida pelo Sistema Especialista. O Gemini apenas interpreta e comunica.


In [ ]:
# Configuração segura da chave da API no Google Colab
# 1. No Colab, clique no ícone de chave/segredos.
# 2. Crie um segredo chamado GEMINI_API_KEY.
# 3. Cole sua chave obtida no Google AI Studio.
# 4. Marque a opção de permitir acesso ao notebook.

try:
    from google.colab import userdata
    gemini_key = userdata.get("GEMINI_API_KEY")
except Exception:
    gemini_key = None

if not gemini_key:
    gemini_key = os.getenv("GEMINI_API_KEY")

if not gemini_key:
    gemini_key = getpass("Cole sua GEMINI_API_KEY apenas para esta execução: ")

os.environ["GEMINI_API_KEY"] = gemini_key

if not os.environ.get("GEMINI_API_KEY"):
    raise RuntimeError("GEMINI_API_KEY não configurada. Configure a chave antes de executar a análise interpretativa.")


In [ ]:
from google import genai

client = genai.Client()
MODELO_GEMINI = "gemini-2.5-flash"


def gerar_analise_interpretativa(log_decisao):
    prompt = f'''
Você é uma camada de análise interpretativa para um sistema AgTech de automação de precisão.

Regras obrigatórias:
- Não altere a decisão técnica do Sistema Especialista.
- Explique em linguagem natural por que a decisão foi tomada.
- Gere alertas estratégicos para o operador de campo.
- Não recomende dosagens químicas específicas.
- Destaque que as ações devem respeitar o Manejo Integrado de Pragas e orientação técnica responsável.

Log do Sistema Especialista:
{json.dumps(log_decisao, ensure_ascii=False, indent=2)}

Responda no formato:
1. Resumo da situação
2. Justificativa da decisão
3. Ação recomendada
4. Alertas estratégicos
5. Próximos dados que devem ser coletados
'''
    resposta = client.models.generate_content(
        model=MODELO_GEMINI,
        contents=prompt
    )
    return resposta.text


analises_gemini = []

for log in logs:
    analise = gerar_analise_interpretativa(log)
    analises_gemini.append({
        "talhao": log["entrada"]["talhao"],
        "analise_gemini": analise
    })

for item in analises_gemini:
    print("\n" + "=" * 80)
    print(f"ANÁLISE INTERPRETATIVA - {item['talhao']}")
    print("=" * 80)
    print(item["analise_gemini"])


## 5. Exportação dos logs

A célula abaixo gera arquivos que podem ser incluídos no repositório GitHub como evidência de funcionamento.


In [ ]:
with open("logs_sistema_especialista.json", "w", encoding="utf-8") as f:
    json.dump(logs, f, ensure_ascii=False, indent=2)

try:
    with open("analises_gemini.json", "w", encoding="utf-8") as f:
        json.dump(analises_gemini, f, ensure_ascii=False, indent=2)
except NameError:
    print("As análises do Gemini ainda não foram geradas.")

df_logs.to_csv("resumo_logs.csv", index=False, encoding="utf-8-sig")

print("Arquivos gerados: logs_sistema_especialista.json, resumo_logs.csv e, se executado, analises_gemini.json")


## 6. Conclusão

O modelo implementado demonstra um agente computacional AgTech capaz de:

- Receber dados simulados de uma Rede Neural de Visão Computacional.
- Aplicar regras de um Sistema Especialista para classificar risco de pragas e risco hídrico.
- Gerar uma decisão operacional baseada em conhecimento explícito.
- Enviar o resultado para o Gemini, que produz uma análise interpretativa em linguagem natural.

Essa arquitetura atende ao critério de que o Gemini não substitui o motor lógico, mas atua como camada de explicação e apoio estratégico.
